# Đo độ dài token trước SFT đầy đủ
Dùng runtime **CPU**. Notebook xác minh source/dữ liệu, rồi đo 512 câu train bằng processor Qwen và collator SFT thật. Không tải model weights, không train GPU. Kết quả chỉ là mẫu cố định, chưa phải kiểm tra toàn bộ 29.459 câu.

In [ ]:
from pathlib import Path

from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Vietnamese_VQA_NLE")
assert ROOT.is_dir(), f"Sai tài khoản Drive hoặc thiếu thư mục dự án: {ROOT}"
print("Drive:", ROOT)

In [ ]:
import hashlib
import importlib.metadata
import json
import subprocess
import sys
import zipfile

SOURCE_DIR = ROOT / "source/releases"
hash_files = sorted(SOURCE_DIR.glob("source_*.sha256"))
assert hash_files, f"Không có source release: {SOURCE_DIR}"
SOURCE_HASH = hash_files[-1]
SOURCE_NAME = SOURCE_HASH.stem
SOURCE_ZIP = SOURCE_DIR / f"{SOURCE_NAME}.zip"
expected, filename = SOURCE_HASH.read_text(encoding="utf-8").split()
assert filename == SOURCE_ZIP.name
digest = hashlib.sha256()
with SOURCE_ZIP.open("rb") as source:
    for block in iter(lambda: source.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == expected, "Source ZIP checksum mismatch"
WORK_SOURCE = Path("/content/vqa_sources") / SOURCE_NAME
REPO = WORK_SOURCE / "Vietnamese-VQA-Explanation"
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    names = archive.namelist()
    assert names and all(
        name.startswith("Vietnamese-VQA-Explanation/") and ".." not in Path(name).parts
        for name in names
    )
    assert archive.testzip() is None
    manifest = json.loads(archive.read("Vietnamese-VQA-Explanation/SOURCE_MANIFEST.json"))
    if REPO.exists():
        assert all(
            hashlib.sha256((REPO / name).read_bytes()).hexdigest() == value
            for name, value in manifest.items()
        ), "Source đã giải nén không khớp"
    else:
        WORK_SOURCE.mkdir(parents=True, exist_ok=False)
        archive.extractall(WORK_SOURCE)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "transformers>=4.49,<5"],
    check=True,
)
print("Source verified:", SOURCE_NAME, expected)
print("Versions:", importlib.metadata.version("torch"), importlib.metadata.version("transformers"))

In [ ]:
BUNDLE = ROOT / "datasets/releases/vivqax_image_disjoint_v1"
EXPECTED_BUNDLE_SHA256 = "46846a81c7b3594d4fc9670ec51019cfd652f1eb169af7978e39dc5a40a55522"
WORK_DATA = Path("/content/vqa_work/vivqax_image_disjoint_v1")
assert hashlib.sha256((BUNDLE / "bundle.json").read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
bundle = json.loads((BUNDLE / "bundle.json").read_text(encoding="utf-8"))
command = [sys.executable, str(REPO / "scripts/verify_coco_bundle.py"), str(BUNDLE)]
if not WORK_DATA.exists():
    command += ["--extract-to", str(WORK_DATA)]
subprocess.run(command, cwd=REPO, check=True)
assert len(list((WORK_DATA / "images").glob("*.jpg"))) == bundle["total_images"]
checksums = WORK_DATA / "metadata/full_image_checksums.csv"
assert hashlib.sha256(checksums.read_bytes()).hexdigest() == bundle["image_checksums_sha256"]
assert (WORK_DATA / "metadata/all.jsonl").is_file()
print("Data verified:", bundle["total_images"], "images")

In [ ]:
RUN_ROOT = ROOT / "runs"
before = {path.name for path in RUN_ROOT.glob("token-profile_*")}
subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO / "scripts/run_pipeline.py"),
        "token-profile",
        "--environment",
        "configs/environments/colab_full.example.yaml",
    ],
    cwd=REPO,
    check=True,
)
created = [path for path in RUN_ROOT.glob("token-profile_*") if path.name not in before]
assert len(created) == 1, f"Không xác định được run mới: {created}"
run = created[0]
execution = json.loads((run / "execution.json").read_text(encoding="utf-8"))
profile = json.loads((run / "token_profile.json").read_text(encoding="utf-8"))
assert execution["status"] == "passed" and execution["returncode"] == 0
assert profile["status"] == "passed" and profile["sample_size"] == 512
assert profile["train_records"] == 29459
print("RUN:", run)
print(json.dumps(profile["summary"], ensure_ascii=False, indent=2))